# 07 — Why Attention? From Static to Contextual Embeddings

**Description:** Discover why one fixed vector per token is not enough, then construct and visualize simple context-dependent updates by hand.
**Level:** Beginner
**Tags:** Language Models, Embeddings, Context, Attention, Visualization

Notebook 03 gave every token an embedding, and Notebook 04 explored their geometry. But a lookup table always returns the same vector for the same token. This notebook isolates the problem that attention will solve: **a token's useful representation should depend on the tokens around it**.

By the end, you will be able to:

- distinguish a static token embedding from a contextual representation;
- explain why ambiguous words expose the limitation of lookup tables;
- construct a transparent context update by hand; and
- interpret movement in a small semantic space.

We will not build queries, keys, or attention weights yet. Those begin in Notebook 08.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. A lookup table has no context

We will use a deliberately interpretable 2D space. The horizontal axis runs from **nature** to **finance**. The vertical axis runs from **object** to **action**. These coordinates are designed by hand for learning; real embeddings are learned and usually have many dimensions.

In [ ]:
embeddings = {
    "bank": np.array([0.0, 0.0]),
    "river": np.array([-1.8, 0.2]),
    "shore": np.array([-1.5, -0.3]),
    "money": np.array([1.8, 0.1]),
    "loan": np.array([1.5, -0.2]),
    "bat": np.array([0.0, -0.3]),
    "cave": np.array([-1.4, -0.7]),
    "flew": np.array([-1.1, 1.6]),
    "baseball": np.array([1.4, -0.8]),
    "swung": np.array([1.0, 1.5]),
}

print("bank in 'river bank':", embeddings["bank"])
print("bank in 'bank loan': ", embeddings["bank"])
assert np.array_equal(embeddings["bank"], embeddings["bank"])

The repeated lookup is not a bug. An embedding table maps a token ID to one row, so `bank` starts at the same point in every sentence. The rest of the model must add context.

## 2. Ambiguity makes the limitation visible

Consider two fragments:

- `river bank` — *bank* means land beside water.
- `bank loan` — *bank* means a financial institution.

The starting vector cannot express both senses at once. A useful representation of `bank` should move toward different evidence in each fragment.

In [ ]:
def plot_vectors(words, title, ax):
    for word in words:
        x, y = embeddings[word]
        ax.scatter(x, y, s=90)
        ax.annotate(word, (x, y), xytext=(5, 5), textcoords="offset points")
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set(xlim=(-2.2, 2.2), ylim=(-1.2, 2.1), xlabel="nature  ←  semantic axis  →  finance", ylabel="object  ←  semantic axis  →  action", title=title)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
plot_vectors(["bank", "river", "shore"], "Water context", axes[0])
plot_vectors(["bank", "money", "loan"], "Financial context", axes[1])
plt.show()

### Predict before running

If `bank` borrowed information from `river`, would its updated x-coordinate become negative or positive? What if it borrowed from `loan`? The next section makes this idea numerical.

## 3. A manual context update

For now, suppose a teacher tells us exactly which context token matters. We update a target by adding a fraction of that context vector:

$$h_{target} = e_{target} + lpha e_{context}$$

Here $e$ is a static embedding, $h$ is a contextual representation, and $lpha$ controls how much context enters. This is not learned attention—it is a transparent sketch of the job attention must perform.

In [ ]:
def contextual_update(target, context, strength=0.75):
    return embeddings[target] + strength * embeddings[context]

bank_near_river = contextual_update("bank", "river")
bank_for_loan = contextual_update("bank", "loan")

print("static bank:       ", embeddings["bank"])
print("bank near river:   ", bank_near_river)
print("bank offering loan:", bank_for_loan)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
start = embeddings["bank"]
ax.scatter(*start, s=120, color="black", label="static bank")
for endpoint, label, color in [
    (bank_near_river, "bank | river context", "#4C78A8"),
    (bank_for_loan, "bank | loan context", "#F58518"),
]:
    delta = endpoint - start
    ax.arrow(*start, *delta, width=0.015, length_includes_head=True, color=color)
    ax.scatter(*endpoint, s=100, color=color, label=label)
ax.set(xlim=(-1.7, 1.7), ylim=(-0.7, 0.7), xlabel="nature  ←  semantic axis  →  finance", ylabel="object  ←  semantic axis  →  action", title="The same starting embedding, two contextual destinations")
ax.legend()
plt.show()

## 4. Context can combine several tokens

A token may need information from more than one position. We can form a weighted average of context vectors, where non-negative weights sum to one:

$$c = \sum_j a_j e_j, \qquad \sum_j a_j = 1$$

Then use $h=e_{target}+c$. The weights below are still chosen by us. Later, the model will compute them.

In [ ]:
def weighted_context(words, weights):
    weights = np.asarray(weights, dtype=float)
    assert len(words) == len(weights)
    assert np.all(weights >= 0) and np.isclose(weights.sum(), 1.0)
    vectors = np.stack([embeddings[word] for word in words])
    return weights @ vectors

water_clues = weighted_context(["river", "shore"], [0.7, 0.3])
finance_clues = weighted_context(["money", "loan"], [0.4, 0.6])

print("water context:  ", water_clues)
print("finance context:", finance_clues)
print("updated bank:   ", embeddings["bank"] + water_clues)
print("updated bank:   ", embeddings["bank"] + finance_clues)

### Your turn: change the weights

Change `[0.7, 0.3]` to `[0.2, 0.8]`. The update stays in the water region, but moves closer to `shore`. This separation is useful:

- the **weights** decide how much to gather from each position;
- the **vectors** determine what information is gathered.

## 5. The same idea resolves *bat*

The token `bat` can refer to an animal or sports equipment. Use one surrounding clue to make each meaning more specific.

In [ ]:
bat_animal = contextual_update("bat", "flew", strength=0.7)
bat_sports = contextual_update("bat", "baseball", strength=0.7)

fig, ax = plt.subplots(figsize=(8, 5))
start = embeddings["bat"]
ax.scatter(*start, s=120, color="black", label="static bat")
for endpoint, label, color in [(bat_animal, "bat | flew", "#54A24B"), (bat_sports, "bat | baseball", "#E45756")]:
    ax.annotate("", xy=endpoint, xytext=start, arrowprops={"arrowstyle": "->", "lw": 2, "color": color})
    ax.scatter(*endpoint, s=100, color=color, label=label)
ax.set(xlim=(-1.3, 1.3), ylim=(-1.1, 1.1), xlabel="nature  ←  semantic axis  →  finance/sport", ylabel="object  ←  semantic axis  →  action", title="Context separates two senses of 'bat'")
ax.legend()
plt.show()

## 6. What must a model learn?

Our hand-written rule hides the hard question: **which tokens should influence each target, and by how much?** A useful mechanism needs to:

1. describe what each target token is looking for;
2. describe what each context token can offer;
3. compare those descriptions to produce relevance scores; and
4. turn scores into mixing weights.

Attention learns this routing. Notebook 08 introduces the first three pieces: queries, keys, and their similarity scores.

## 7. Challenges

1. Create two contexts for another ambiguous word such as `crane`, `spring`, or `mouse`. Invent 2D vectors and plot both updates.
2. Set a context weight to 1. What does the weighted average become?
3. Try weights that do not sum to one. Why does that change both direction and scale?
4. Explain why contextualization does not require changing the embedding-table row itself.

## Takeaways

- An embedding lookup is **static**: the same token ID always starts with the same vector.
- A **contextual representation** combines that starting vector with information from surrounding tokens.
- Ambiguous words make the need for context easy to see, but every token can benefit from context.
- Weighted sums provide a simple way to gather information from several positions.
- We chose the weights manually. Attention will compute them from the tokens themselves.